<a href="https://colab.research.google.com/github/louistrue/learn-ifc/blob/main/BFH-25-Tabbed-Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# IFC Tabbed Dashboard Example

This notebook creates a minimal [Dash](https://dash.plotly.com/) application inside Jupyter that helps students explore parsed IFC data.
The dashboard is organised into tabs that focus on two common analysis tasks:

1. How building elements are distributed across storeys.
2. How materials are used across storeys and element categories.

The example uses the `Duplex_A_20110907.ifc` model that is already part of this repository, but you can adapt the data-loading step to any IFC file.


## 1. Install dependencies (Colab)
If you are running this notebook on Google Colab or another ephemeral environment, install the required packages. Skip this cell on a local machine where the dependencies are already available.


In [ ]:
%pip install ifcopenshell pandas plotly dash jupyter-dash


## 2. Parse IFC data and prepare analysis tables

The helper functions below extract three key pieces of information from the IFC model:

* The building storey for each element (`IfcRelContainedInSpatialStructure`).
* The element type (e.g. `IfcWall`, `IfcDoor`).
* Any materials associated with the element (`IfcRelAssociatesMaterial`).

The resulting tidy `pandas` DataFrame contains one row per element-material combination, making it easy to aggregate counts for different dashboard views.


In [ ]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path

import pandas as pd

try:
    import ifcopenshell
except ImportError as exc:
    raise ImportError("ifcopenshell is required for this notebook. Install it via `pip install ifcopenshell`.") from exc

# Path to the sample IFC file shipped with this repository.
IFC_PATH = Path('Modelle/Duplex_A_20110907.ifc')

if not IFC_PATH.exists():
    raise FileNotFoundError(f"Could not find IFC file at {IFC_PATH.resolve()}. Adjust IFC_PATH to point to your model.")

model = ifcopenshell.open(IFC_PATH)


def storey_label(storey):
    """Return a readable label for a building storey."""
    if storey is None:
        return 'Not assigned'
    return storey.LongName or storey.Name or f"Storey {storey.id()}"


# Map each element (by GlobalId) to its containing storey.
storey_by_element: dict[str, str] = {}
for relation in model.by_type('IfcRelContainedInSpatialStructure'):
    structure = relation.RelatingStructure
    if not structure or not structure.is_a('IfcBuildingStorey'):
        continue
    label = storey_label(structure)
    for element in relation.RelatedElements:
        storey_by_element[element.GlobalId] = label


def material_names(material) -> list[str]:
    """Return a list of material names for different IFC material types."""
    if material is None:
        return ['Unspecified']
    if material.is_a('IfcMaterial'):
        return [material.Name or 'Unnamed material']
    if material.is_a('IfcMaterialList'):
        return [mat.Name or 'Unnamed material' for mat in material.Materials]
    if material.is_a('IfcMaterialLayerSetUsage'):
        return material_names(material.ForLayerSet)
    if material.is_a('IfcMaterialLayerSet'):
        names = []
        for layer in material.MaterialLayers:
            if layer.Material:
                names.append(layer.Material.Name or 'Unnamed material')
        return names or ['Unspecified']
    if material.is_a('IfcMaterialConstituentSet'):
        names = []
        for constituent in material.MaterialConstituents:
            if constituent.Material:
                names.append(constituent.Material.Name or 'Unnamed material')
        return names or ['Unspecified']
    if material.is_a('IfcMaterialProfileSetUsage'):
        return material_names(material.ForProfileSet)
    if material.is_a('IfcMaterialProfileSet'):
        names = []
        for profile in material.MaterialProfiles:
            if profile.Material:
                names.append(profile.Material.Name or 'Unnamed material')
        return names or ['Unspecified']
    fallback = getattr(material, "Name", None)
    return [fallback or 'Unspecified']


# Map each element (by GlobalId) to the materials applied to it.
materials_by_element: dict[str, list[str]] = defaultdict(list)
for relation in model.by_type('IfcRelAssociatesMaterial'):
    names = material_names(relation.RelatingMaterial)
    for element in relation.RelatedObjects:
        materials_by_element[element.GlobalId].extend(names)

# Remove duplicated material names per element
for element_id, names in materials_by_element.items():
    deduplicated: list[str] = []
    for name in names:
        if name not in deduplicated:
            deduplicated.append(name)
    materials_by_element[element_id] = deduplicated

records: list[dict[str, str]] = []
for element in model.by_type('IfcElement'):
    element_id = element.GlobalId
    element_type = element.is_a()
    storey = storey_by_element.get(element_id, 'Not assigned')
    element_materials = materials_by_element.get(element_id, ['Unspecified'])
    for material in element_materials:
        records.append({
            'ElementId': element_id,
            'ElementType': element_type,
            'Storey': storey,
            'Material': material or 'Unspecified',
        })

if not records:
    raise ValueError('No element records found. Ensure the IFC model contains elements and materials.')

ifc_df = pd.DataFrame(records)
ifc_df.head()


### Aggregated datasets

The dashboard will use a few pre-computed aggregations. These remain accessible as standalone DataFrames so that students can inspect and reuse them.


In [ ]:
storey_summary = (
    ifc_df.groupby('Storey')['ElementId']
    .nunique()
    .reset_index(name='ElementCount')
    .sort_values('ElementCount', ascending=False)
)

storey_element_breakdown = (
    ifc_df.groupby(['Storey', 'ElementType'])['ElementId']
    .nunique()
    .reset_index(name='ElementCount')
    .sort_values(['Storey', 'ElementCount'], ascending=[True, False])
)

storey_material_breakdown = (
    ifc_df.groupby(['Storey', 'Material'])['ElementId']
    .nunique()
    .reset_index(name='ElementCount')
    .sort_values(['Storey', 'ElementCount'], ascending=[True, False])
)

material_element_breakdown = (
    ifc_df.groupby(['Material', 'ElementType'])['ElementId']
    .nunique()
    .reset_index(name='ElementCount')
    .sort_values('ElementCount', ascending=False)
)

storey_summary.head()


## 3. Build the tabbed Dash dashboard

The `JupyterDash` helper spins up a Dash server that renders directly inside the notebook.
Each tab focuses on one question and includes small text summaries alongside interactive Plotly charts.
Feel free to customise the layout, styling, or add new components.


In [ ]:
from dash import dash_table, dcc, html
from jupyter_dash import JupyterDash
import plotly.express as px

app = JupyterDash(__name__)

storey_total_fig = px.bar(
    storey_summary,
    x='Storey',
    y='ElementCount',
    text_auto='.0f',
    title='Elements per building storey',
    color='ElementCount',
    color_continuous_scale='Blues'
)
storey_total_fig.update_layout(coloraxis_showscale=False, xaxis_title='', yaxis_title='Number of elements')

storey_type_fig = px.bar(
    storey_element_breakdown,
    x='Storey',
    y='ElementCount',
    color='ElementType',
    title='Element type distribution per storey',
    text_auto='.0f'
)
storey_type_fig.update_layout(xaxis_title='', yaxis_title='Number of elements', legend_title='Element type')

material_storey_fig = px.bar(
    storey_material_breakdown,
    x='Storey',
    y='ElementCount',
    color='Material',
    title='Material usage per storey',
    text_auto='.0f'
)
material_storey_fig.update_layout(xaxis_title='', yaxis_title='Elements using material', legend_title='Material')

material_heatmap_fig = px.density_heatmap(
    material_element_breakdown,
    x='ElementType',
    y='Material',
    z='ElementCount',
    title='Material vs element type',
    color_continuous_scale='Viridis'
)
material_heatmap_fig.update_layout(xaxis_title='Element type', yaxis_title='Material')

top_material_combinations = material_element_breakdown.head(15)

app.layout = html.Div(
    [
        html.H1('IFC Elements and Materials Explorer'),
        html.P(
            'Use the tabs to explore how elements and materials are distributed across the building. '
            'This starter dashboard can be adapted to any IFC dataset by editing the data preparation cell above.'
        ),
        dcc.Tabs(
            [
                dcc.Tab(
                    label='Storey overview',
                    children=[
                        html.H3('How many elements exist per storey?'),
                        html.P('Counts are based on unique element GlobalIds assigned to each building storey.'),
                        dcc.Graph(figure=storey_total_fig),
                        html.Hr(),
                        html.H3('Which element types dominate each storey?'),
                        html.P('Stacked bars highlight the element categories present on each storey.'),
                        dcc.Graph(figure=storey_type_fig),
                    ],
                ),
                dcc.Tab(
                    label='Material insights',
                    children=[
                        html.H3('Which materials are used on each storey?'),
                        html.P('Materials are derived from IfcRelAssociatesMaterial and summarised per storey.'),
                        dcc.Graph(figure=material_storey_fig),
                        html.Hr(),
                        html.H3('Material vs element type'),
                        html.P('Use the heatmap to spot dominant material-element combinations.'),
                        dcc.Graph(figure=material_heatmap_fig),
                        html.Hr(),
                        html.H3('Top material & element combinations'),
                        dash_table.DataTable(
                            data=top_material_combinations.to_dict('records'),
                            columns=[{'name': c, 'id': c} for c in top_material_combinations.columns],
                            style_table={'maxHeight': '400px', 'overflowY': 'auto'},
                            style_cell={'padding': '0.5rem', 'textAlign': 'left'},
                            style_header={'fontWeight': 'bold'},
                        ),
                    ],
                ),
            ]
        ),
    ]
)

app.run_server(mode='inline', height=900, port=8051, dev_tools_ui=False)


## 4. Next steps

* Swap in another IFC model by updating `IFC_PATH` and re-running the data preparation cells.
* Add filters (e.g. `dcc.Dropdown`) so users can focus on specific storeys or materials.
* Export aggregated tables as CSV for downstream reporting.
* Combine with geometry viewers (e.g. `ifcopenshell.geom`) to link the plots with 3D previews.
